# Model Inspection
Load a checkpoint and visualise predictions on the training set (ground truth available).

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CHECKPOINT = "checkpoints/vit_depth_transformer_pretrainedTrue/best.pth"
DATA_ROOT  = "data/"          # must contain train/*_rgb.png and train/*_depth.npy
TRAIN_DIR  = "train"

# IDs to inspect — integers, set to None for random
# Example: IDS = [1, 42, 1337]
IDS = None
N_RANDOM = 10

IMG_SIZE = None   # None → read from checkpoint

In [ ]:
import os, sys, random, glob
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T

sys.path.insert(0, os.path.abspath("."))
from src.models.model import DepthModel
from src.metrics import si_rmse

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]
normalize = T.Normalize(mean=_MEAN, std=_STD)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
ckpt = torch.load(CHECKPOINT, map_location=device)
cfg  = ckpt["cfg"]
img_size = IMG_SIZE or cfg["model"]["img_size"]

model = DepthModel(
    decoder_type   = cfg["model"]["decoder_type"],
    pretrained     = False,
    img_size       = img_size,
    patch_size     = cfg["model"]["patch_size"],
    decoder_blocks = cfg["model"]["decoder_blocks"],
    embed_dim      = cfg["model"]["embed_dim"],
    num_heads      = cfg["model"]["num_heads"],
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Loaded epoch {ckpt['epoch']}  val_si_rmse={ckpt.get('val_si_rmse', float('nan')):.4f}")
print(f"img_size={img_size}  decoder={cfg['model']['decoder_type']}")

In [ ]:
# ── Resolve sample IDs ────────────────────────────────────────────────────────
train_dir = os.path.join(DATA_ROOT, TRAIN_DIR)
all_rgb   = sorted(glob.glob(os.path.join(train_dir, "*_rgb.png")))
print(f"Found {len(all_rgb)} training images")

def path_to_int_id(path):
    # train_000001_rgb.png → 1
    return int(os.path.basename(path).split("_")[1])

def int_id_to_str(i):
    return f"{i:06d}"

if IDS is not None:
    id_set   = set(IDS)
    selected = [p for p in all_rgb if path_to_int_id(p) in id_set]
    missing  = id_set - {path_to_int_id(p) for p in selected}
    if missing:
        print(f"WARNING: IDs not found: {missing}")
else:
    selected = random.sample(all_rgb, min(N_RANDOM, len(all_rgb)))

print(f"Inspecting {len(selected)} samples: {[path_to_int_id(p) for p in selected]}")

In [ ]:
# ── Run inference ─────────────────────────────────────────────────────────────
def load_sample(rgb_path, img_size):
    depth_path = rgb_path.replace("_rgb.png", "_depth.npy")
    image = Image.open(rgb_path).convert("RGB")
    image = image.resize((img_size, img_size), Image.LANCZOS)
    depth_gt = np.load(depth_path).astype(np.float32)
    if depth_gt.shape[0] != img_size:
        depth_gt = np.array(Image.fromarray(depth_gt).resize((img_size, img_size), Image.NEAREST))
    tensor = normalize(TF.to_tensor(image)).unsqueeze(0)
    return np.array(image), depth_gt, tensor

results = []
with torch.no_grad():
    for path in selected:
        rgb_np, gt, tensor = load_sample(path, img_size)
        pred = model(tensor.to(device)).squeeze().cpu().numpy()
        metric = si_rmse(
            torch.from_numpy(pred).unsqueeze(0).unsqueeze(0),
            torch.from_numpy(gt).unsqueeze(0).unsqueeze(0),
        ).item()
        results.append({"id": path_to_int_id(path), "rgb": rgb_np, "gt": gt, "pred": pred, "si_rmse": metric})

mean_rmse = np.mean([r["si_rmse"] for r in results])
print(f"Mean SI-RMSE: {mean_rmse:.4f}")

In [ ]:
# ── Visualise ─────────────────────────────────────────────────────────────────
CMAP = "plasma"

for r in results:
    fig = plt.figure(figsize=(14, 4))
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.05)

    # RGB
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(r["rgb"])
    ax0.set_title(f"RGB  (id={r['id']})", fontsize=9)
    ax0.axis("off")

    # GT depth
    vmin, vmax = r["gt"].min(), r["gt"].max()
    ax1 = fig.add_subplot(gs[1])
    im1 = ax1.imshow(r["gt"], cmap=CMAP, vmin=vmin, vmax=vmax)
    ax1.set_title("GT depth", fontsize=9)
    ax1.axis("off")
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # Predicted depth
    ax2 = fig.add_subplot(gs[2])
    im2 = ax2.imshow(r["pred"], cmap=CMAP, vmin=vmin, vmax=vmax)
    ax2.set_title(f"Pred depth  SI-RMSE={r['si_rmse']:.4f}", fontsize=9)
    ax2.axis("off")
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Absolute error
    ax3 = fig.add_subplot(gs[3])
    err = np.abs(r["pred"] - r["gt"])
    im3 = ax3.imshow(err, cmap="hot")
    ax3.set_title("Abs error", fontsize=9)
    ax3.axis("off")
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    plt.suptitle(f"id={r['id']}  SI-RMSE={r['si_rmse']:.4f}", fontsize=10, y=1.02)
    plt.show()

print(f"\nMean SI-RMSE over {len(results)} samples: {mean_rmse:.4f}")